In [ ]:
import copy
import csv
import json
import os
import random
from pathlib import Path

import flatbuffers
import litert_torch
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from ai_edge_litert import schema_py_generated as schema
from ai_edge_litert.interpreter import Interpreter, OpResolverType
from ai_edge_quantizer import quantizer, recipe
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from transformers import MobileViTForImageClassification

SEED = 42


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


set_seed(SEED)

In [ ]:
!pwd

# Test 1 setup

In [ ]:
PATH = Path("/home/jovyan/work/UFSC/butterflies_austria/Dataset_train_test_split/butterflies-austria")
NOTEBOOK_DIR = Path.cwd()
CHECKPOINT_PATH = NOTEBOOK_DIR / "best_mobilevit_xx_small_v0.pt"
OUTPUT_DIR = NOTEBOOK_DIR / "test1_results"
OUTPUT_DIR.mkdir(exist_ok=True)

BATCH_SIZE = 64
IMG_SIZE = 256
RESIZE_SIZE = 288
CALIBRATION_SAMPLES_PER_CLASS = 50
GROUP_COUNTS = {1: 8, 2: 4, 5: 4}
MIN_GROUPED_INT8_ACCURACY = 0.70
MIN_ACCURACY_GAIN = 0.50
EQUIVALENCE_MAX_ERROR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(CHECKPOINT_PATH)

print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

# Dataset

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize(RESIZE_SIZE),
    transforms.RandomCrop(IMG_SIZE),
    transforms.Lambda(lambda image: image.convert("RGB")),
    transforms.RandomHorizontalFlip(p=0.1),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Lambda(lambda image: image[[2, 1, 0]]),
])

val_test_transforms = transforms.Compose([
    transforms.Resize(RESIZE_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.Lambda(lambda image: image.convert("RGB")),
    transforms.ToTensor(),
    transforms.Lambda(lambda image: image[[2, 1, 0]]),
])

In [ ]:
train_dir = PATH / "train"
val_dir = PATH / "val"
test_dir = PATH / "test"

train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
calibration_source = datasets.ImageFolder(train_dir, transform=val_test_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=val_test_transforms)
test_dataset = datasets.ImageFolder(test_dir, transform=val_test_transforms)

class_names = train_dataset.classes
class_to_idx = train_dataset.class_to_idx

if calibration_source.class_to_idx != class_to_idx:
    raise ValueError("Training and calibration classes are not aligned")

if val_dataset.class_to_idx != class_to_idx:
    raise ValueError("Training and validation classes are not aligned")

if test_dataset.class_to_idx != class_to_idx:
    raise ValueError("Training and test classes are not aligned")

loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": 4,
    "pin_memory": DEVICE.type == "cuda",
    "persistent_workers": True,
}

val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)

print(f"Classes: {len(class_names)}")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
train_targets = np.asarray(train_dataset.targets)
train_counts = np.bincount(train_targets, minlength=len(class_names))

print("Training distribution:")
for name, count in zip(class_names, train_counts):
    print(f"  {name:30s}: {count}")

In [ ]:
def select_stratified_calibration_indices(targets, samples_per_class, seed):
    generator = torch.Generator().manual_seed(seed)
    selected_indices = []

    for label in range(len(class_names)):
        class_indices = np.flatnonzero(targets == label)

        if len(class_indices) < samples_per_class:
            raise ValueError(f"Class {class_names[label]} has insufficient calibration samples")

        permutation = torch.randperm(len(class_indices), generator=generator)[:samples_per_class]
        selected_indices.extend(class_indices[permutation.numpy()].tolist())

    return sorted(selected_indices)


calibration_indices = select_stratified_calibration_indices(
    np.asarray(calibration_source.targets),
    CALIBRATION_SAMPLES_PER_CLASS,
    SEED,
)
calibration_dataset = Subset(calibration_source, calibration_indices)
calibration_loader = DataLoader(calibration_dataset, shuffle=False, **loader_options)

with (OUTPUT_DIR / "calibration_manifest.csv").open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["path", "label", "class_name"])
    writer.writeheader()

    for index in calibration_indices:
        path, label = calibration_source.samples[index]
        writer.writerow({"path": path, "label": label, "class_name": class_names[label]})

calibration_counts = np.bincount(
    np.asarray(calibration_source.targets)[calibration_indices],
    minlength=len(class_names),
)

if not np.all(calibration_counts == CALIBRATION_SAMPLES_PER_CLASS):
    raise RuntimeError("Calibration dataset is not balanced")

print(f"Calibration samples: {len(calibration_dataset)}")
print(f"Samples per class: {calibration_counts.tolist()}")

# Model and checkpoint

In [ ]:
MODEL_NAME = "apple/mobilevit-xx-small"

hf_model = MobileViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(class_names),
    id2label={index: name for index, name in enumerate(class_names)},
    label2id={name: index for index, name in enumerate(class_names)},
    classifier_dropout_prob=0.3,
    ignore_mismatched_sizes=True,
)


class MobileViTClassifier(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, inputs):
        return self.model(pixel_values=inputs, return_dict=False)[0]


model = MobileViTClassifier(hf_model).to(DEVICE)

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint, strict=True)
model.eval()

total_params = sum(parameter.numel() for parameter in model.parameters())
print(f"Checkpoint loaded: {CHECKPOINT_PATH.name}")
print(f"Total parameters: {total_params:,}")

# Activation diagnostics

In [ ]:
def find_inverted_residuals(root):
    blocks = []

    for parent_name, parent in root.named_modules():
        for child_name, child in parent.named_children():
            required_layers = ("expand_1x1", "conv_3x3", "reduce_1x1")

            if all(hasattr(child, layer_name) for layer_name in required_layers):
                full_name = f"{parent_name}.{child_name}" if parent_name else child_name
                blocks.append({
                    "parent": parent,
                    "child_name": child_name,
                    "name": full_name,
                    "module": child,
                })

    if len(blocks) != 7:
        raise RuntimeError(f"Expected 7 inverted residual blocks, found {len(blocks)}")

    return blocks


def collect_channel_statistics(current_model, blocks, data_loader, device):
    accumulators = {}
    handles = []

    def create_hook(block_index):
        def hook(_, __, output):
            values = output.detach()
            channel_count = values.shape[1]
            current = accumulators.get(block_index)

            if current is None:
                current = {
                    "minimum": torch.full((channel_count,), float("inf")),
                    "maximum": torch.full((channel_count,), float("-inf")),
                    "sum_squares": torch.zeros(channel_count, dtype=torch.float64),
                    "element_count": 0,
                }
                accumulators[block_index] = current

            dimensions = (0, 2, 3)
            current["minimum"] = torch.minimum(
                current["minimum"],
                values.amin(dim=dimensions).float().cpu(),
            )
            current["maximum"] = torch.maximum(
                current["maximum"],
                values.amax(dim=dimensions).float().cpu(),
            )
            current["sum_squares"] += values.float().square().sum(dim=dimensions).double().cpu()
            current["element_count"] += values.shape[0] * values.shape[2] * values.shape[3]

        return hook

    for block_index, block_info in enumerate(blocks, start=1):
        handles.append(block_info["module"].expand_1x1.register_forward_hook(create_hook(block_index)))

    current_model.eval()

    try:
        with torch.no_grad():
            for images, _ in data_loader:
                current_model(images.to(device, non_blocking=True))
    finally:
        for handle in handles:
            handle.remove()

    records = []
    scales_by_block = {}

    for block_index, block_info in enumerate(blocks, start=1):
        current = accumulators[block_index]
        minimum = current["minimum"]
        maximum = current["maximum"]
        channel_range = maximum - minimum
        channel_scale = torch.clamp(channel_range / 255.0, min=torch.finfo(torch.float32).eps)
        tensor_scale = max(
            float((maximum.max() - minimum.min()) / 255.0),
            torch.finfo(torch.float32).eps,
        )
        rms = torch.sqrt(current["sum_squares"] / current["element_count"]).float()
        scales_by_block[block_index] = channel_scale

        for channel in range(len(channel_scale)):
            approximate_codes = min(256, int(torch.floor(channel_range[channel] / tensor_scale).item()) + 1)
            records.append({
                "block": block_index,
                "block_name": block_info["name"],
                "channel": channel,
                "minimum": float(minimum[channel]),
                "maximum": float(maximum[channel]),
                "rms": float(rms[channel]),
                "channel_scale": float(channel_scale[channel]),
                "tensor_scale": tensor_scale,
                "rms_over_tensor_scale": float(rms[channel] / tensor_scale),
                "approximate_int8_codes": approximate_codes,
            })

    return records, scales_by_block


def write_csv(path, records):
    if not records:
        raise ValueError(f"No records to write: {path}")

    with path.open("w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=records[0].keys())
        writer.writeheader()
        writer.writerows(records)

In [ ]:
inverted_residuals = find_inverted_residuals(model)
channel_records, scales_by_block = collect_channel_statistics(
    model,
    inverted_residuals,
    calibration_loader,
    DEVICE,
)
write_csv(OUTPUT_DIR / "channel_statistics.csv", channel_records)



In [ ]:
groups_by_block = {}
group_records = []

for block_index, group_count in GROUP_COUNTS.items():
    ordered_channels = torch.argsort(scales_by_block[block_index])
    groups = [group.tolist() for group in torch.tensor_split(ordered_channels, group_count)]
    groups_by_block[block_index] = groups

    for group_index, channels in enumerate(groups):
        group_scales = scales_by_block[block_index][channels]
        group_records.append({
            "block": block_index,
            "group": group_index,
            "channels": json.dumps(channels),
            "channel_count": len(channels),
            "minimum_scale": float(group_scales.min()),
            "maximum_scale": float(group_scales.max()),
            "scale_ratio": float(group_scales.max() / group_scales.min()),
        })

write_csv(OUTPUT_DIR / "group_statistics.csv", group_records)

with (OUTPUT_DIR / "group_configuration.json").open("w") as file:
    json.dump({str(index): groups for index, groups in groups_by_block.items()}, file, indent=2)



In [ ]:
for block_index, scale_values in scales_by_block.items():
    ratio = float(scale_values.max() / scale_values.min())
    print(f"Block {block_index}: channels={len(scale_values)} scale_ratio={ratio:.2f}x")

# Structural grouping

In [ ]:
class SlicedConvLayer(nn.Module):
    def __init__(self, source_layer, channels, depthwise):
        super().__init__()
        source_conv = source_layer.convolution
        channel_indices = torch.as_tensor(channels, dtype=torch.long)

        if depthwise:
            in_channels = len(channels)
            out_channels = len(channels)
            groups = len(channels)
            weight = source_conv.weight.detach().index_select(0, channel_indices)
            bias = None if source_conv.bias is None else source_conv.bias.detach().index_select(0, channel_indices)
        else:
            in_channels = source_conv.in_channels
            out_channels = len(channels)
            groups = source_conv.groups
            weight = source_conv.weight.detach().index_select(0, channel_indices)
            bias = None if source_conv.bias is None else source_conv.bias.detach().index_select(0, channel_indices)

        self.convolution = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=source_conv.kernel_size,
            stride=source_conv.stride,
            padding=source_conv.padding,
            dilation=source_conv.dilation,
            groups=groups,
            bias=bias is not None,
            padding_mode=source_conv.padding_mode,
        )

        with torch.no_grad():
            self.convolution.weight.copy_(weight)

            if bias is not None:
                self.convolution.bias.copy_(bias)

        self.normalization = self.create_batch_norm(source_layer.normalization, channel_indices)
        self.activation = copy.deepcopy(source_layer.activation)

    @staticmethod
    def create_batch_norm(source, channel_indices):
        if source is None:
            return None

        if not isinstance(source, nn.BatchNorm2d):
            raise TypeError(f"Unsupported normalization: {type(source).__name__}")

        result = nn.BatchNorm2d(
            len(channel_indices),
            eps=source.eps,
            momentum=source.momentum,
            affine=source.affine,
            track_running_stats=source.track_running_stats,
        )

        with torch.no_grad():
            if source.affine:
                result.weight.copy_(source.weight.detach().index_select(0, channel_indices))
                result.bias.copy_(source.bias.detach().index_select(0, channel_indices))

            if source.track_running_stats:
                result.running_mean.copy_(source.running_mean.detach().index_select(0, channel_indices))
                result.running_var.copy_(source.running_var.detach().index_select(0, channel_indices))
                result.num_batches_tracked.copy_(source.num_batches_tracked.detach())

        return result

    def forward(self, inputs):
        outputs = self.convolution(inputs)

        if self.normalization is not None:
            outputs = self.normalization(outputs)

        if self.activation is not None:
            outputs = self.activation(outputs)

        return outputs


def fold_conv_batch_norm(source_layer):
    convolution = source_layer.convolution
    normalization = source_layer.normalization
    weight = convolution.weight.detach().clone()
    bias = (
        convolution.bias.detach().clone()
        if convolution.bias is not None
        else torch.zeros(convolution.out_channels, dtype=weight.dtype, device=weight.device)
    )

    if normalization is None:
        return weight, bias

    if not isinstance(normalization, nn.BatchNorm2d):
        raise TypeError(f"Unsupported normalization: {type(normalization).__name__}")

    factor = normalization.weight.detach() / torch.sqrt(normalization.running_var.detach() + normalization.eps)
    folded_weight = weight * factor[:, None, None, None]
    folded_bias = (bias - normalization.running_mean.detach()) * factor + normalization.bias.detach()
    return folded_weight, folded_bias


class GroupedBranch(nn.Module):
    def __init__(self, source_block, channels, group_count, folded_weight, folded_bias):
        super().__init__()
        channel_indices = torch.as_tensor(channels, dtype=torch.long)
        self.expand = SlicedConvLayer(source_block.expand_1x1, channels, depthwise=False)
        self.depthwise = SlicedConvLayer(source_block.conv_3x3, channels, depthwise=True)
        source_reduce = source_block.reduce_1x1.convolution
        self.reduce = nn.Conv2d(
            in_channels=len(channels),
            out_channels=source_reduce.out_channels,
            kernel_size=source_reduce.kernel_size,
            stride=source_reduce.stride,
            padding=source_reduce.padding,
            dilation=source_reduce.dilation,
            groups=source_reduce.groups,
            bias=True,
            padding_mode=source_reduce.padding_mode,
        )

        with torch.no_grad():
            self.reduce.weight.copy_(folded_weight.index_select(1, channel_indices))
            self.reduce.bias.copy_(folded_bias / group_count)

    def forward(self, inputs):
        outputs = self.expand(inputs)
        outputs = self.depthwise(outputs)
        return self.reduce(outputs)


class GroupedInvertedResidual(nn.Module):
    def __init__(self, source_block, groups):
        super().__init__()
        folded_weight, folded_bias = fold_conv_batch_norm(source_block.reduce_1x1)
        self.branches = nn.ModuleList([
            GroupedBranch(source_block, channels, len(groups), folded_weight, folded_bias)
            for channels in groups
        ])
        self.use_residual = source_block.use_residual

    def forward(self, inputs):
        outputs = self.branches[0](inputs)

        for branch in self.branches[1:]:
            outputs = outputs + branch(inputs)

        return inputs + outputs if self.use_residual else outputs


def build_grouped_model(reference_model, groups):
    grouped_model = copy.deepcopy(reference_model).cpu().eval()
    grouped_blocks = find_inverted_residuals(grouped_model)

    for block_index, block_groups in groups.items():
        block_info = grouped_blocks[block_index - 1]
        grouped_block = GroupedInvertedResidual(block_info["module"], block_groups)
        setattr(block_info["parent"], block_info["child_name"], grouped_block)

    return grouped_model

In [ ]:
reference_model = copy.deepcopy(model).cpu().eval()
grouped_model = build_grouped_model(reference_model, groups_by_block)

print("Grouped blocks:")
for block_index, group_count in GROUP_COUNTS.items():
    print(f"  block={block_index} groups={group_count}")

In [ ]:
def collect_torch_outputs(current_model, data_loader, device):
    outputs = []
    labels = []
    current_model = current_model.to(device).eval()

    with torch.no_grad():
        for images, targets in data_loader:
            outputs.append(current_model(images.to(device, non_blocking=True)).cpu())
            labels.append(targets.cpu())

    current_model.cpu()
    return torch.cat(outputs), torch.cat(labels)


reference_outputs, validation_labels = collect_torch_outputs(reference_model, val_loader, DEVICE)
grouped_outputs, grouped_validation_labels = collect_torch_outputs(grouped_model, val_loader, DEVICE)

if not torch.equal(validation_labels, grouped_validation_labels):
    raise RuntimeError("Validation labels changed during equivalence evaluation")

reference_predictions = reference_outputs.argmax(dim=1)
grouped_predictions = grouped_outputs.argmax(dim=1)
reference_accuracy = float((reference_predictions == validation_labels).float().mean())
grouped_accuracy = float((grouped_predictions == validation_labels).float().mean())
top1_agreement = float((reference_predictions == grouped_predictions).float().mean())
maximum_error = float((reference_outputs - grouped_outputs).abs().max())
mean_error = float((reference_outputs - grouped_outputs).abs().mean())
logits_cosine = float(torch.nn.functional.cosine_similarity(
    reference_outputs.flatten(),
    grouped_outputs.flatten(),
    dim=0,
))

equivalence = {
    "reference_accuracy": reference_accuracy,
    "grouped_accuracy": grouped_accuracy,
    "top1_agreement": top1_agreement,
    "maximum_logit_error": maximum_error,
    "mean_logit_error": mean_error,
    "logits_cosine": logits_cosine,
}

with (OUTPUT_DIR / "fp32_equivalence.json").open("w") as file:
    json.dump(equivalence, file, indent=2)

print(json.dumps(equivalence, indent=2))



In [ ]:
if top1_agreement != 1.0 or reference_accuracy != grouped_accuracy or maximum_error > EQUIVALENCE_MAX_ERROR:
    raise RuntimeError("Grouped model failed FP32 equivalence")

# Exportação e validação para ESP32-S3

A exportação usa pooling espacial fixo, incorpora buffers externos e rejeita artefatos incompatíveis com o schema e os kernels disponíveis no firmware.

In [ ]:
ORIGINAL_FP32_PATH = OUTPUT_DIR / "mobilevit_original_fp32.tflite"
ORIGINAL_INT8_PATH = OUTPUT_DIR / "mobilevit_original_stratified_int8.tflite"
GROUPED_FP32_PATH = OUTPUT_DIR / "mobilevit_g8_4_4_fp32.tflite"
GROUPED_INT8_PATH = OUTPUT_DIR / "mobilevit_g8_4_4_stratified_int8.tflite"
MAX_RESOLVER_ACCURACY_DELTA = 0.01

ESP32_INPUT_SHAPE = (1, 3, 256, 256)
ESP32_OUTPUT_SHAPE = (1, len(class_names))
ESP32_SUPPORTED_OP_VERSIONS = {
    "ADD": frozenset({1}),
    "AVERAGE_POOL_2D": frozenset({1}),
    "CONCATENATION": frozenset({1}),
    "CONV_2D": frozenset({1}),
    "DEPTHWISE_CONV_2D": frozenset({1}),
    "FULLY_CONNECTED": frozenset({5}),
    "LOGISTIC": frozenset({1}),
    "MEAN": frozenset({1}),
    "MUL": frozenset({1}),
    "PAD": frozenset({1}),
    "QUANTIZE": frozenset({1}),
    "RESHAPE": frozenset({1}),
    "RSQRT": frozenset({1}),
    "SOFTMAX": frozenset({1}),
    "SQUARED_DIFFERENCE": frozenset({1}),
    "SUB": frozenset({1}),
    "TRANSPOSE": frozenset({1}),
}
ESP32_SUPPORTED_OPS = frozenset(ESP32_SUPPORTED_OP_VERSIONS)


class MobileViTFixedAvgPool(nn.Module):
    def __init__(self, source_model):
        super().__init__()
        classifier_model = source_model.model
        self.mobilevit = classifier_model.mobilevit
        self.dropout = classifier_model.dropout
        self.classifier = classifier_model.classifier

    def forward(self, inputs):
        features = self.mobilevit.conv_stem(inputs)
        features = self.mobilevit.encoder(
            features,
            output_hidden_states=False,
            return_dict=False,
        )[0]
        if not self.mobilevit.expand_output:
            raise RuntimeError("MobileViT sem expansão final não suportada")

        features = self.mobilevit.conv_1x1_exp(features)
        spatial_shape = tuple(features.shape[-2:])
        if spatial_shape != (8, 8):
            raise ValueError(f"Feature map esperado: 8x8; recebido: {spatial_shape}")

        features = F.avg_pool2d(features, kernel_size=(8, 8), stride=1)
        features = torch.flatten(features, 1)
        features = self.dropout(features)
        return self.classifier(features)


def mobilevit_export_adapter(source_model):
    candidate = copy.deepcopy(source_model).cpu().eval()
    return MobileViTFixedAvgPool(candidate).eval()


def prepare_export_model(source_model, inputs):
    baseline = copy.deepcopy(source_model).cpu().eval()
    candidate = mobilevit_export_adapter(source_model)

    with torch.no_grad():
        expected = baseline(inputs[0])
        actual = candidate(inputs[0])

    torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)
    max_error = float(torch.max(torch.abs(expected - actual)))
    print(f"Erro absoluto máximo do adaptador: {max_error:.10f}")
    return candidate


def inline_external_buffers(source_path, target_path):
    content = source_path.read_bytes()
    model = schema.Model.GetRootAsModel(content, 0)
    model_object = schema.ModelT.InitFromObj(model)
    converted = 0

    for index, target_buffer in enumerate(model_object.buffers):
        source_buffer = model.Buffers(index)
        offset = source_buffer.Offset()
        size = source_buffer.Size()
        if not offset or not size:
            continue

        end = offset + size
        if end > len(content):
            raise ValueError(f"Buffer {index} fora do arquivo: {offset}:{end}")

        target_buffer.data = bytearray(content[offset:end])
        target_buffer.offset = 0
        target_buffer.size = 0
        converted += 1

    builder = flatbuffers.Builder(len(content))
    model_offset = model_object.Pack(builder)
    builder.Finish(model_offset, file_identifier=b"TFL3")
    target_path.write_bytes(builder.Output())
    return converted


def export_with_inline_buffers(exporter, output_path):
    external_path = output_path.with_name(f"{output_path.stem}_external{output_path.suffix}")
    inline_path = output_path.with_name(f".{output_path.name}.tmp")
    external_path.unlink(missing_ok=True)
    inline_path.unlink(missing_ok=True)

    try:
        exporter(external_path)
        converted = inline_external_buffers(external_path, inline_path)
        os.replace(inline_path, output_path)
    finally:
        external_path.unlink(missing_ok=True)
        inline_path.unlink(missing_ok=True)

    print(f"{output_path.name}: {converted} buffers incorporados")


def export_fp32(current_model, output_path, sample_inputs):
    with torch.no_grad():
        edge_model = litert_torch.convert(current_model, sample_inputs)

    export_with_inline_buffers(
        lambda external_path: edge_model.export(str(external_path)),
        output_path,
    )
    print(f"Exported: {output_path} ({output_path.stat().st_size / 1024 / 1024:.2f} MB)")


sample_image = val_dataset[0][0]
sample_inputs = (sample_image.unsqueeze(0).cpu(),)
reference_export_model = prepare_export_model(reference_model, sample_inputs)
grouped_export_model = prepare_export_model(grouped_model, sample_inputs)

In [ ]:
def create_calibration_data(model_path, dataset):
    interpreter = Interpreter(model_path=str(model_path))
    signatures = interpreter.get_signature_list()

    if len(signatures) != 1:
        raise ValueError(f"Expected one signature, found: {list(signatures)}")

    signature_name, signature = next(iter(signatures.items()))
    input_names = signature["inputs"]

    if len(input_names) != 1:
        raise ValueError(f"Expected one input, found: {input_names}")

    input_name = input_names[0]
    samples = []

    for image, _ in dataset:
        samples.append({input_name: image.unsqueeze(0).numpy().astype(np.float32)})

    return {signature_name: samples}

In [ ]:
export_fp32(reference_export_model, ORIGINAL_FP32_PATH, sample_inputs)

In [ ]:
export_fp32(grouped_export_model, GROUPED_FP32_PATH, sample_inputs)

In [ ]:
def quantize_static(fp32_path, output_path, dataset):
    calibration_data = create_calibration_data(fp32_path, dataset)
    current_quantizer = quantizer.Quantizer(str(fp32_path))
    current_quantizer.load_quantization_recipe(recipe.static_wi8_ai8())
    calibration_result = current_quantizer.calibrate(calibration_data)
    result = current_quantizer.quantize(calibration_result)

    export_with_inline_buffers(
        lambda external_path: result.export_model(str(external_path), overwrite=True),
        output_path,
    )
    print(f"Quantized: {output_path} ({output_path.stat().st_size / 1024 / 1024:.2f} MB)")

In [ ]:
quantize_static(ORIGINAL_FP32_PATH, ORIGINAL_INT8_PATH, calibration_dataset)

In [ ]:
quantize_static(GROUPED_FP32_PATH, GROUPED_INT8_PATH, calibration_dataset)

In [ ]:
def create_interpreter(model_path, resolver_type):
    interpreter = Interpreter(
        model_path=str(model_path),
        experimental_op_resolver_type=resolver_type,
    )
    interpreter.allocate_tensors()
    return interpreter


def external_buffer_indices(model):
    return [
        index
        for index in range(model.BuffersLength())
        if model.Buffers(index).Offset() or model.Buffers(index).Size()
    ]


def operator_versions(model):
    names = {
        value: name
        for name, value in vars(schema.BuiltinOperator).items()
        if name.isupper() and isinstance(value, int)
    }
    return [
        (
            names.get(
                model.OperatorCodes(index).BuiltinCode(),
                str(model.OperatorCodes(index).BuiltinCode()),
            ),
            model.OperatorCodes(index).Version(),
        )
        for index in range(model.OperatorCodesLength())
    ]


def inspect_tflite(model_path):
    content = model_path.read_bytes()
    if len(content) < 8 or content[4:8] != b"TFL3":
        raise RuntimeError(f"{model_path.name}: identificador TFL3 inválido")

    model = schema.Model.GetRootAsModel(content, 0)
    if model.Version() != 3:
        raise RuntimeError(f"{model_path.name}: schema {model.Version()} não suportado")

    external_buffers = external_buffer_indices(model)
    if external_buffers:
        raise RuntimeError(f"{model_path.name}: buffers externos {external_buffers}")

    interpreter = create_interpreter(
        model_path,
        OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES,
    )
    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]
    operators = [operation["op_name"] for operation in interpreter._get_ops_details()]
    versions = operator_versions(model)
    input_shape = tuple(int(value) for value in input_detail["shape"])
    output_shape = tuple(int(value) for value in output_detail["shape"])

    if input_shape != ESP32_INPUT_SHAPE:
        raise RuntimeError(f"{model_path.name}: entrada {input_shape}, esperado {ESP32_INPUT_SHAPE}")
    if output_shape != ESP32_OUTPUT_SHAPE:
        raise RuntimeError(f"{model_path.name}: saída {output_shape}, esperado {ESP32_OUTPUT_SHAPE}")
    if "SUM" in operators:
        raise RuntimeError(f"{model_path.name}: operador SUM não permitido")

    unsupported = sorted(set(operators) - ESP32_SUPPORTED_OPS)
    unsupported_versions = [
        [name, version]
        for name, version in versions
        if version not in ESP32_SUPPORTED_OP_VERSIONS.get(name, frozenset())
    ]
    float_tensors = [
        detail["name"]
        for detail in interpreter.get_tensor_details()
        if detail["dtype"] in (np.float16, np.float32, np.float64)
    ]
    strict_full_int8 = (
        input_detail["dtype"] == np.int8
        and output_detail["dtype"] == np.int8
        and not float_tensors
        and "DEQUANTIZE" not in operators
        and "CUSTOM" not in operators
        and not any(name.startswith("Flex") for name in operators)
    )
    result = {
        "model": model_path.name,
        "schema": model.Version(),
        "external_buffers": external_buffers,
        "input_dtype": np.dtype(input_detail["dtype"]).name,
        "output_dtype": np.dtype(output_detail["dtype"]).name,
        "input_shape": list(input_shape),
        "output_shape": list(output_shape),
        "input_quantization": list(input_detail["quantization"]),
        "output_quantization": list(output_detail["quantization"]),
        "size_bytes": model_path.stat().st_size,
        "strict_full_int8": strict_full_int8,
        "unsupported_operators": unsupported,
        "unsupported_operator_versions": unsupported_versions,
    }
    return result, operators

In [ ]:
original_inspection, original_operators = inspect_tflite(ORIGINAL_INT8_PATH)
grouped_inspection, grouped_operators = inspect_tflite(GROUPED_INT8_PATH)

with (OUTPUT_DIR / "tflite_inspection.json").open("w") as file:
    json.dump({"original": original_inspection, "grouped": grouped_inspection}, file, indent=2)

with (OUTPUT_DIR / "grouped_operators.csv").open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["index", "operator"])
    writer.writeheader()
    writer.writerows({"index": index, "operator": name} for index, name in enumerate(grouped_operators))

print(json.dumps({"original": original_inspection, "grouped": grouped_inspection}, indent=2))



In [ ]:
if not grouped_inspection["strict_full_int8"]:
    raise RuntimeError("Grouped model is not strict full INT8")

if grouped_inspection["unsupported_operators"]:
    raise RuntimeError(
        f"Operadores sem kernel validado no ESP32: {grouped_inspection['unsupported_operators']}"
    )

if grouped_inspection["unsupported_operator_versions"]:
    raise RuntimeError(
        "Versões de operadores não validadas no ESP32: "
        f"{grouped_inspection['unsupported_operator_versions']}"
    )

In [ ]:
def evaluate_tflite(
    model_path,
    data_loader,
    resolver_type=OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES,
):
    interpreter = create_interpreter(model_path, resolver_type)
    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]
    input_scale, input_zero_point = input_detail["quantization"]
    output_scale, output_zero_point = output_detail["quantization"]
    targets = []
    predictions = []

    for images, labels in data_loader:
        for image, label in zip(images.numpy(), labels.numpy()):
            values = np.expand_dims(image, axis=0)

            if np.issubdtype(input_detail["dtype"], np.integer):
                if input_scale == 0:
                    raise ValueError(f"Invalid input quantization scale in {model_path}")

                limits = np.iinfo(input_detail["dtype"])
                values = np.round(values / input_scale + input_zero_point)
                values = np.clip(values, limits.min, limits.max).astype(input_detail["dtype"])
            else:
                values = values.astype(input_detail["dtype"])

            interpreter.set_tensor(input_detail["index"], values)
            interpreter.invoke()
            output = interpreter.get_tensor(output_detail["index"])

            if np.issubdtype(output_detail["dtype"], np.integer) and output_scale > 0:
                output = (output.astype(np.float32) - output_zero_point) * output_scale

            targets.append(int(label))
            predictions.append(int(np.argmax(output, axis=1)[0]))

    metrics = {
        "accuracy": float(accuracy_score(targets, predictions)),
        "macro_precision": float(precision_score(targets, predictions, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(targets, predictions, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(targets, predictions, average="weighted", zero_division=0)),
    }
    return metrics, np.asarray(targets), np.asarray(predictions)

In [ ]:
validation_models = {
    "original_fp32": ORIGINAL_FP32_PATH,
    "original_int8": ORIGINAL_INT8_PATH,
    "grouped_fp32": GROUPED_FP32_PATH,
    "grouped_int8": GROUPED_INT8_PATH,
}
validation_results = []
validation_predictions = {}

for model_name, model_path in validation_models.items():
    metrics, targets, predictions = evaluate_tflite(model_path, val_loader)
    validation_predictions[model_name] = predictions
    validation_results.append({"model": model_name, "split": "validation", **metrics})
    print(model_name, metrics)

reference_metrics, reference_targets, reference_predictions = evaluate_tflite(
    GROUPED_INT8_PATH,
    val_loader,
    OpResolverType.BUILTIN_REF,
)
if not np.array_equal(validation_target, reference_targets):
    raise RuntimeError("Resolver reference changed validation targets")

builtin_predictions = validation_predictions["grouped_int8"]
resolver_accuracy_delta = abs(
    next(row["accuracy"] for row in validation_results if row["model"] == "grouped_int8")
    - reference_metrics["accuracy"]
)
resolver_agreement = float(np.mean(builtin_predictions == reference_resolver_predictions))
if resolver_accuracy_delta > MAX_RESOLVER_ACCURACY_DELTA:
    raise RuntimeError(
        f"Divergência excessiva entre resolvers: {resolver_accuracy_delta:.6f}"
    )
if np.unique(reference_resolver_predictions).size == 1:
    raise RuntimeError("Resolver reference produziu saída constante")

original_int8_accuracy = next(row["accuracy"] for row in validation_results if row["model"] == "original_int8")
grouped_int8_accuracy = next(row["accuracy"] for row in validation_results if row["model"] == "grouped_int8")
validation_gain = grouped_int8_accuracy - original_int8_accuracy
test_gate_passed = (
    grouped_int8_accuracy >= MIN_GROUPED_INT8_ACCURACY
    and validation_gain >= MIN_ACCURACY_GAIN
)

print(f"Grouped INT8 validation gain: {validation_gain:.4f}")
print(f"Resolver accuracy delta: {resolver_accuracy_delta:.6f}")
print(f"Resolver prediction agreement: {resolver_agreement:.6f}")
print(f"Test gate passed: {test_gate_passed}")

In [ ]:
result_rows = list(validation_results)

if test_gate_passed:
    for model_name, model_path in validation_models.items():
        metrics, test_targets, test_predictions = evaluate_tflite(model_path, test_loader)
        result_rows.append({"model": model_name, "split": "test", **metrics})
        print(model_name, metrics)

        if model_name == "grouped_int8":
            report = classification_report(
                test_targets,
                test_predictions,
                target_names=class_names,
                digits=4,
                zero_division=0,
                output_dict=True,
            )

            with (OUTPUT_DIR / "grouped_int8_test_report.json").open("w") as file:
                json.dump(report, file, indent=2)

            np.savetxt(
                OUTPUT_DIR / "grouped_int8_test_confusion_matrix.csv",
                confusion_matrix(test_targets, test_predictions, labels=range(len(class_names))),
                delimiter=",",
                fmt="%d",
            )
else:
    print("Test evaluation skipped because the validation gate failed")

In [ ]:
with (OUTPUT_DIR / "results.csv").open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=result_rows[0].keys())
    writer.writeheader()
    writer.writerows(result_rows)

validation_report = classification_report(
    validation_target,
    validation_predictions["grouped_int8"],
    target_names=class_names,
    digits=4,
    zero_division=0,
    output_dict=True,
)

with (OUTPUT_DIR / "grouped_int8_validation_report.json").open("w") as file:
    json.dump(validation_report, file, indent=2)

np.savetxt(
    OUTPUT_DIR / "grouped_int8_validation_confusion_matrix.csv",
    confusion_matrix(
        validation_target,
        validation_predictions["grouped_int8"],
        labels=range(len(class_names)),
    ),
    delimiter=",",
    fmt="%d",
)

print(f"Results saved to: {OUTPUT_DIR}")

In [ ]:
for row in result_rows:
    print(
        f"{row['split']:10s} {row['model']:15s} "
        f"accuracy={row['accuracy']:.4f} macro_f1={row['macro_f1']:.4f}"
    )

In [ ]:
print("Test 1 completed")